# ESG sentence classification on Kaggle GPU

Classifies Vietnamese/English annual-report & news sentences with **`nguyen599/ViDeBERTa-v3-ESG-base`** (DeBERTa-v2, multi-label).

**Before running:**
1. Sidebar &rarr; **Notebook options** &rarr; *Accelerator* = **GPU T4** (NOT P100 — the image's PyTorch dropped P100/sm_60 support), *Internet* = **On** (needed to download the model the first time).
2. **Add data**: upload a **folder** of `*.jsonl` files (produced locally by `python -m data_processing.prepare_sentences ...`, e.g. `data/interim/sentences/` or `data/interim/esg_news/`) as a Kaggle **Dataset**, then attach it. It mounts at `/kaggle/input/<your-dataset-slug>/`.
3. Set `INPUT_DIR` in the config cell to that folder. Every `*.jsonl` under it is classified.

**Why sigmoid, not softmax:** the model's `config.problem_type == "multi_label_classification"`, so each label (Environmental / Social / Governance / Neutral) gets an independent sigmoid score.

**How labels are decided (tuned on the AAA report):**
- A sentence is tagged with a pillar when that pillar's score &ge; `THRESHOLD` (**0.45**).
- A sentence is **ESG-relevant** when `Neutral < NEUTRAL_THRESHOLD` (**0.5**), *not* when a single pillar clears the bar. In practice this model spreads probability like a softmax and rarely tags two pillars at once, so a clearly-ESG sentence can have its signal split across two pillars (e.g. labour-law compliance &rarr; S=0.44, G=0.44) with neither clearing the threshold. Keying `esg` off Neutral recovers those cases.

**Input/Output:** point `INPUT_DIR` at a folder of `*.jsonl`; results are written to `OUTPUT_DIR` (default `/kaggle/working/classified/`), one file per input named `<stem>_classified.jsonl` (e.g. `aaa_sentences.jsonl` &rarr; `aaa_sentences_classified.jsonl`). Download the whole folder from `/kaggle/working/`.

In [ ]:
# 1) Dependencies. torch + CUDA already ship on Kaggle GPU images.
!pip install -q "transformers==4.51.0" sentencepiece protobuf

In [ ]:
# 2) Config -- point INPUT_DIR at a FOLDER of .jsonl files (one per news article / report).
import glob, os

MODEL_ID     = "nguyen599/ViDeBERTa-v3-ESG-base"
INPUT_DIR    = "/kaggle/input/aaa-esg-sentences"   # <-- EDIT ME: folder containing *.jsonl
OUTPUT_DIR   = "/kaggle/working/classified"        # output folder; one *_classified.jsonl per input
BATCH_SIZE   = 64
MAX_LENGTH   = 256   # sentences are short; model cap is 512
THRESHOLD    = 0.45  # sigmoid cutoff for assigning an E/S/G pillar tag
NEUTRAL_THRESHOLD = 0.5  # ESG-relevant when the Neutral score is below this
ESG_PILLARS  = ("Environmental", "Social", "Governance")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Collect every .jsonl under INPUT_DIR (searched recursively).
input_files = sorted(glob.glob(os.path.join(INPUT_DIR, "**", "*.jsonl"), recursive=True))
assert input_files, f"No .jsonl found under {INPUT_DIR} -- check the path / attach your dataset."
print(f"{len(input_files)} input file(s) found under {INPUT_DIR}:")
for p in input_files:
    print("  ", p)

In [ ]:
# 3) Load the model on GPU.
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "(no GPU)")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID).to(device).eval()

def _norm(lbl):
    return "Neutral" if lbl.strip().lower() in {"neural", "neutral", "none"} else lbl

id2label = {int(i): _norm(l) for i, l in model.config.id2label.items()}
print("problem_type:", model.config.problem_type, "| labels:", id2label)

In [ ]:
# 4) Helper to read one .jsonl (or .csv) file of sentences.
import csv, json

def read_sentences(path):
    if path.lower().endswith(".csv"):
        with open(path, encoding="utf-8-sig", newline="") as f:
            return list(csv.DictReader(f))
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

In [ ]:
# 5) Batched multi-label inference (sigmoid), as a reusable function.
#    Pillar tags = E/S/G scores >= THRESHOLD.
#    esg flag    = Neutral < NEUTRAL_THRESHOLD (robust to signal split across pillars).
def classify_texts(texts):
    preds = []
    for start in range(0, len(texts), BATCH_SIZE):
        batch = texts[start:start + BATCH_SIZE]
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=MAX_LENGTH, return_tensors="pt").to(device)
        with torch.no_grad():
            probs = torch.sigmoid(model(**enc).logits).cpu().tolist()
        for row in probs:
            scores = {id2label[i]: round(float(p), 4) for i, p in enumerate(row)}
            labels = [p for p in ESG_PILLARS if scores.get(p, 0.0) >= THRESHOLD]
            esg = scores.get("Neutral", 0.0) < NEUTRAL_THRESHOLD
            preds.append({"scores": scores, "labels": labels, "esg": esg})
    return preds

In [ ]:
# 6) Classify every input file -> OUTPUT_DIR/<stem>_classified.jsonl
#    e.g. aaa_sentences.jsonl  ->  classified/aaa_sentences_classified.jsonl
last_merged = []
grand_counts = {l: 0 for l in ESG_PILLARS}
grand_total = grand_esg = 0

for path in input_files:
    rows = read_sentences(path)
    texts = [str(r.get("text", "")) for r in rows]
    preds = classify_texts(texts)
    merged = [{**r, **p} for r, p in zip(rows, preds)]
    last_merged = merged

    stem = os.path.splitext(os.path.basename(path))[0]
    out_path = os.path.join(OUTPUT_DIR, f"{stem}_classified.jsonl")
    with open(out_path, "w", encoding="utf-8") as f:
        for r in merged:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    counts = {l: 0 for l in ESG_PILLARS}
    esg_total = 0
    for r in merged:
        for l in r["labels"]:
            counts[l] += 1
        esg_total += int(r["esg"])
    grand_total += len(merged)
    grand_esg += esg_total
    for l in ESG_PILLARS:
        grand_counts[l] += counts[l]
    print(f"{stem}: {len(merged)} sentences -> {out_path} | "
          f"ESG {esg_total}/{len(merged)} | pillars {counts}")

print(f"\nDone. {len(input_files)} file(s) written to {OUTPUT_DIR}/")
print(f"Totals: ESG-relevant {grand_esg}/{grand_total} | pillar tag counts: {grand_counts}")

In [ ]:
# 7) (Optional) eyeball a few results per pillar from the LAST processed file.
import itertools
for pillar in ESG_PILLARS:
    print(f"\n=== {pillar} ===")
    sample = (r for r in last_merged if pillar in r["labels"])
    for r in itertools.islice(sample, 3):
        print(f"  p{r.get('page')} [{r['scores'][pillar]:.2f}] {r['text'][:120]}")